# Titanic Survival Prediction

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder
import joblib
import os

## 2. Load Data

In [ ]:
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')
combine = [train_df, test_df]

## 3. EDA & Preprocessing

In [ ]:
# Check missing
print(train_df.isnull().sum())

for dataset in combine:
    # Handle Missing
    dataset['Age'] = dataset['Age'].fillna(dataset['Age'].median())
    dataset['Embarked'] = dataset['Embarked'].fillna(dataset['Embarked'].mode()[0])
    dataset['Fare'] = dataset['Fare'].fillna(test_df['Fare'].dropna().median())
    
    # Feature Engineering
    dataset['FamilySize'] = dataset['SibSp'] + dataset['Parch'] + 1
    
    # Extract Title
    dataset['Title'] = dataset.Name.str.extract(' ([A-Za-z]+)\.', expand=False)
    dataset['Title'] = dataset['Title'].replace(['Lady', 'Countess','Capt', 'Col',\
        'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    dataset['Title'] = dataset['Title'].replace('Mlle', 'Miss')
    dataset['Title'] = dataset['Title'].replace('Ms', 'Miss')
    dataset['Title'] = dataset['Title'].replace('Mme', 'Mrs')

## 4. Encoding

In [ ]:
le_sex = LabelEncoder()
le_embarked = LabelEncoder()
le_title = LabelEncoder()

for dataset in combine:
    dataset['Sex'] = le_sex.fit_transform(dataset['Sex'])
    dataset['Embarked'] = le_embarked.fit_transform(dataset['Embarked'])
    dataset['Title'] = le_title.fit_transform(dataset['Title'])
    
    dataset.drop(['Cabin', 'Name', 'Ticket'], axis=1, inplace=True)

X_train = train_df.drop(['Survived', 'PassengerId'], axis=1)
Y_train = train_df['Survived']
X_test  = test_df.drop('PassengerId', axis=1).copy()

## 5. Model Training

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(n_neighbors=3),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

X_tr, X_val, Y_tr, Y_val = train_test_split(X_train, Y_train, test_size=0.2, random_state=42)

best_model = None
best_acc = 0

for name, model in models.items():
    model.fit(X_tr, Y_tr)
    Y_pred = model.predict(X_val)
    acc = accuracy_score(Y_val, Y_pred)
    print(f"{name} Accuracy: {acc:.4f}")
    if acc > best_acc:
        best_acc = acc
        best_model = model

## 6. Save Model and Submission

In [ ]:
best_model.fit(X_train, Y_train)
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/best_model.pkl')

Y_pred_submission = best_model.predict(X_test)
submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": Y_pred_submission
})
submission.to_csv('../submission.csv', index=False)